# Custom QM Dataset (PyTorch)

The `QMDataset` class is a base class for quantum mechanical datasets. It generates graph properties
from XYZ files, which store atomic coordinates. Bond information can be optionally generated via
RDKit or OpenBabel.

This tutorial demonstrates:
1. Creating a custom `QMDataset` subclass
2. Loading quantum chemistry data (coordinates, energies)
3. Building distance-based graphs with `set_range` preprocessor
4. Converting to PyG Data objects
5. Training a SchNet model on the dataset

## 0. Create Example Data

We create an artificial set of XYZ files and a CSV with labels.

In [ ]:
import os
import numpy as np

os.makedirs("ExampleQM", exist_ok=True)
os.makedirs("ExampleQM/XYZ_files", exist_ok=True)

xyz_list = [
    "4\n\nC 0.9949499369 0.0 0.0\nC 2.1952097416 0.0 0.0\nH -0.0707100034 0.0 0.0\nH 3.2608699799 0.0 0.0\n",
    "5\n\nC 0.9982599616 -0.0024599999 -0.0043599997\nH 2.0902099609 -0.0024299999 0.0041399999\nH 0.6337899566 1.0268599987 0.0041399999\nH 0.6270399690 -0.5277299881 0.8781099916\nH 0.6413599849 -0.5074699521 -0.9053999186\n",
    "3\n\nO 0.00000 0.00000 0.11779\nH 0.00000 0.75545 -0.47116\nH 0.00000 -0.75545 -0.47116"
]

# Write individual XYZ files
for i, x in enumerate(xyz_list):
    with open(f"ExampleQM/XYZ_files/mol_{i}.xyz", "w") as f:
        f.write(x)

# Write combined XYZ file
xyz_data = "".join(xyz_list)
with open("ExampleQM/qm.xyz", "w") as f:
    f.write(xyz_data)

# CSV with file names and energy labels
csv_info = "ID,files,energy\n0,mol_0.xyz,-13.0\n1,mol_1.xyz,-20.0\n2,mol_2.xyz,-34.0"
with open("ExampleQM/qm.csv", "w") as f:
    f.write(csv_info)

print("Files created.")

Expected file structure:

```
ExampleQM/
    qm.csv
    qm.xyz
    XYZ_files/
        mol_0.xyz
        mol_1.xyz
        mol_2.xyz
    qm.sdf         # Created by prepare_data()
```

## 1. Initialization

`QMDataset` requires the data directory, the XYZ or CSV file name, and optionally a subdirectory for individual files.

In [ ]:
from kgcnn_torch.data.qm import QMDataset

dts = QMDataset(
    file_name="qm.xyz",
    file_directory="XYZ_files",
    data_directory="ExampleQM",
    dataset_name="ExampleQM"
)

## 2. Data Preparation

`prepare_data()` consolidates individual XYZ files into one combined file and optionally generates
an SDF file with bond information (via RDKit or OpenBabel).

In [ ]:
dts.prepare_data(
    overwrite=True,
    file_column_name="files",
    make_sdf=True
)

## 3. Read Data into Memory

`read_in_memory()` reads structures from the SDF file (with bond info) or falls back to the XYZ file.
It also reads labels from the CSV.

In [ ]:
dts.read_in_memory(label_column_name="energy")

print(f"Number of graphs: {len(dts)}")
print(f"Properties of graph 0: {list(dts[0].keys())}")

## 4. Inspect the Dataset

In [ ]:
# Check coordinates
print("Coordinates:")
for i, coords in enumerate(dts.obtain_property("node_coordinates")):
    print(f"  Molecule {i}: {coords.shape} atoms")

# Check symbols
print("\nSymbols:", dts.obtain_property("node_symbol"))

# Check labels
print("\nLabels:", dts.obtain_property("graph_labels"))

In [ ]:
# Check bond (edge) information from SDF
print("Edge indices:", dts.obtain_property("edge_indices"))
print("Edge numbers (bond orders):", dts.obtain_property("edge_number"))

## 5. Build Distance-Based Graph

For geometric GNN models like SchNet, we build edges based on pairwise distances rather than chemical bonds.
The `set_range` preprocessor creates `range_indices` and `range_attributes` (distances) based on a
cutoff radius and maximum number of neighbours.

In [ ]:
dts.map_list(
    "set_range",
    max_distance=2.0,
    max_neighbours=15,
    do_invert_distance=False,
    self_loops=False,
    exclusive=True
)

print("Range indices:", dts.obtain_property("range_indices"))
print("Range attributes (distances):", dts.obtain_property("range_attributes"))

In [ ]:
# Clean dataset: remove any graphs with empty or missing properties
removed = dts.clean(inputs=["edge_number", "edge_indices"])
print(f"Removed {len(removed)} invalid graphs.")

## 6. Angle Features (Optional)

For models like DimeNet that use angular information, the `set_angle` preprocessor
computes angle triplet indices and angle values from the range connections.

In [ ]:
dts.map_list(
    "set_angle",
    allow_multi_edges=False,
    compute_angles=True
)

print("Angle indices (edge pairs):", dts.obtain_property("angle_indices")[:1])
print("Angle values:", dts.obtain_property("angle_attributes")[:1])

## 7. Convert to PyG and Train SchNet

We convert the dataset to PyG Data objects using `to_pyg_list()`,
which swaps the edge index convention from KGCNN (target, source) to PyG (source, target).

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

# Convert to PyG. Use range_indices as edges for the geometric model.
pyg_list = dts.to_pyg_list(
    node_key="node_number",
    pos_key="node_coordinates",
    edge_key="range_indices",
    label_key="graph_labels"
)

print(f"Number of PyG graphs: {len(pyg_list)}")
print(f"Example: {pyg_list[0]}")

In [ ]:
from kgcnn_torch.models.schnet import SchNetModel

model = SchNetModel(
    node_dim=32,
    depth=3,
    units=64,
    gauss_bins=20,
    gauss_distance=4.0,
    gauss_sigma=0.4,
    last_mlp_units=[64, 32],
    num_targets=1,
    output_embedding="graph",
    make_distance=True,
    expand_distance=True
)

print(model)

In [ ]:
from kgcnn_torch.training.trainer import fit

train_loader = DataLoader(pyg_list, batch_size=2, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=train_loader,  # Same data for demo only
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-3),
    loss_fn=nn.MSELoss(),
    epochs=50,
    device=device,
    metrics={"mae": lambda pred, target: torch.mean(torch.abs(pred - target))},
    verbose=1
)

print(f"Final train loss: {history['train_loss'][-1]:.4f}")

## 8. Extensive Scaler (for Atomization Energies)

For extensive targets like total energies, it is helpful to remove per-element energy offsets.
The `QMGraphLabelScaler` fits per-element offsets and scales the labels.

In [ ]:
from kgcnn_torch.data.transform import StandardScaler

# Standard scaling of labels
labels = np.array([g.obtain_property("graph_labels") for g in dts])
scaler = StandardScaler()
labels_scaled = scaler.fit_transform(labels.reshape(-1, 1))

print("Original labels:", labels)
print("Scaled labels:", labels_scaled.flatten())
print("Inverse:", scaler.inverse_transform(labels_scaled).flatten())

## Summary

This notebook demonstrated the full workflow for using `QMDataset` in kgcnn-torch:

1. **Create data** -- XYZ files + CSV with labels
2. **prepare_data()** -- Consolidate files and generate bond info
3. **read_in_memory()** -- Load coordinates, symbols, labels
4. **map_list("set_range")** -- Build distance-based edges
5. **map_list("set_angle")** -- Compute angle features (optional)
6. **to_pyg_list()** -- Convert to PyG Data
7. **fit()** -- Train a SchNet model with the PyTorch trainer